# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu and rename sequences according to convention

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai/Documents/Avian_Flu_Files/"
originals = downloads + "avian-influenza/avian-influenza_Files/"
temp_files = downloads + "avian-influenza_Temp_Files/"
complete_files = downloads + "avian-influenza_Complete_Files/"

os.chdir(downloads)

In [2]:
# Read metadata

metadata_folder = downloads + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# Find only >= 2024 using run ID from metadata
metadata["Collection_Date"] = pd.to_numeric(metadata["Collection_Date"], errors='coerce')
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats



C:\Users\maksiaevai\AppData\Local\Temp\1\ipykernel_18424\542306095.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats


In [3]:
display(metadata_new)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,ReleaseDate,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-008354-001-original,SRP503016,H5N1,NaN,SRS21079812,False,NaN
1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-005-original,SRP503016,H5N1,NaN,SRS21079811,False,NaN
2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-004-original,SRP503016,H5N1,NaN,SRS21079810,False,NaN
3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-003-original,SRP503016,H5N1,NaN,SRS21079809,False,NaN
4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,2024-04-20 18:20:39,2024-04-20T18:12:00Z,1,24-009108-002-original,SRP503016,H5N1,NaN,SRS21079808,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7392,SRR32633103,WGS,148.75,108436208,PRJNA1102327,SAMN47290837,Viral,40656883,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:13,1,25-005512-003,SRP503016,NaN,"MILK, BULK TANK",SRS24304291,False,NaN
7393,SRR32633104,WGS,148.77,90195365,PRJNA1102327,SAMN47290836,Viral,33878271,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:13,1,25-005512-002,SRP503016,NaN,"MILK, BULK TANK",SRS24304283,False,NaN
7394,SRR32633105,WGS,148.64,89418970,PRJNA1102327,SAMN47290835,Viral,33592911,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:15,1,25-005512-001,SRP503016,NaN,"MILK, BULK TANK",SRS24304286,False,NaN
7395,SRR32633106,WGS,148.58,84745077,PRJNA1102327,SAMN47290826,Viral,32103811,USDA-NVSL,2025,...,2025-03-10 14:39:02,2025-03-10 14:32:17,1,25-005501-002,SRP503016,NaN,"MILK, BULK TANK",SRS24304281,False,NaN


In [4]:
# Naming convention
# >A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]
# host_type is from manual animal reference
# In metadata, we have: host, geo_loc_name, isolate, year
# We need: host_type, genotype

# host = Host
# geo_loc_name = geo_loc_name
# isolate = isolate
# collection date = Collection_Date
# serotype = serotype
# host type = [from ref] -- use 
# genotype = [from genoflu] -- use output.tsv

os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_anderson(metadata_new, animals_ref) # Get host type
years = metadata_new["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

# Get genotype from genoflu
os.chdir(temp_files)
output_tsv = pd.read_csv("output.tsv", delimiter="\t")

b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})

metadata_new = metadata_new.merge(b313_and_d11_only[["Run", "Genotype"]])

print(metadata_new)

c:\Users\maksiaevai\Documents\Avian_Flu\utils.py:206: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata["Host_Type"] = animal_types


              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR28752446        WGS      146.11   93605195  PRJNA1102327   
1     SRR28752446        WGS      146.11   93605195  PRJNA1102327   
2     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
3     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
4     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
...           ...        ...         ...        ...           ...   
6262  SRR32633103        WGS      148.75  108436208  PRJNA1102327   
6263  SRR32633104        WGS      148.77   90195365  PRJNA1102327   
6264  SRR32633105        WGS      148.64   89418970  PRJNA1102327   
6265  SRR32633106        WGS      148.58   84745077  PRJNA1102327   
6266  SRR32633107        WGS      148.50  114370864  PRJNA1102327   

         BioSample BioSampleModel     Bytes Center Name  Collection_Date  ...  \
0     SAMN41019184          Viral  30074178   USDA-NVSL             2024  ...   
1     SAM

In [5]:
# Make names

names = ">A/" + metadata_new["Host"] + "/" + metadata_new["geo_loc_name"] + "/" + metadata_new["isolate"] + "/" + years + "|H5N1|" + metadata_new["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata_new["Host_Type"] + "|" + metadata_new["Genotype"]

metadata_new["Name"] = names

display(metadata_new)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,Host_Type,Genotype,Name
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,24-008354-001-original,SRP503016,H5N1,NaN,SRS21079812,False,NaN,avian,B3.13,>A/Blackbird/USA/24-008354-001-original/2024|H...
1,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,24-008354-001-original,SRP503016,H5N1,NaN,SRS21079812,False,NaN,avian,B3.13,>A/Blackbird/USA/24-008354-001-original/2024|H...
2,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,24-009108-005-original,SRP503016,H5N1,NaN,SRS21079811,False,NaN,cattle,B3.13,>A/Cattle/USA/24-009108-005-original/2024|H5N1...
3,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,24-009108-005-original,SRP503016,H5N1,NaN,SRS21079811,False,NaN,cattle,B3.13,>A/Cattle/USA/24-009108-005-original/2024|H5N1...
4,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,24-009108-004-original,SRP503016,H5N1,NaN,SRS21079810,False,NaN,cattle,B3.13,>A/Cattle/USA/24-009108-004-original/2024|H5N1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6262,SRR32633103,WGS,148.75,108436208,PRJNA1102327,SAMN47290837,Viral,40656883,USDA-NVSL,2025,...,25-005512-003,SRP503016,NaN,"MILK, BULK TANK",SRS24304291,False,NaN,cattle,B3.13,>A/CATTLE/USA/25-005512-003/2025|H5N1|2025|cat...
6263,SRR32633104,WGS,148.77,90195365,PRJNA1102327,SAMN47290836,Viral,33878271,USDA-NVSL,2025,...,25-005512-002,SRP503016,NaN,"MILK, BULK TANK",SRS24304283,False,NaN,cattle,B3.13,>A/CATTLE/USA/25-005512-002/2025|H5N1|2025|cat...
6264,SRR32633105,WGS,148.64,89418970,PRJNA1102327,SAMN47290835,Viral,33592911,USDA-NVSL,2025,...,25-005512-001,SRP503016,NaN,"MILK, BULK TANK",SRS24304286,False,NaN,cattle,B3.13,>A/CATTLE/USA/25-005512-001/2025|H5N1|2025|cat...
6265,SRR32633106,WGS,148.58,84745077,PRJNA1102327,SAMN47290826,Viral,32103811,USDA-NVSL,2025,...,25-005501-002,SRP503016,NaN,"MILK, BULK TANK",SRS24304281,False,NaN,cattle,B3.13,>A/CATTLE/USA/25-005501-002/2025|H5N1|2025|cat...


In [ ]:
# Make fasta files

fasta_folder = downloads + "avian-influenza/fasta/"

os.chdir(fasta_folder)

pairs = []
fasta_files = {}

for genotype in ["B3.13", "D1.1"]:
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        pair = genotype + "_" + segment
        pairs.append(pair)

for run in metadata_new["Run"].values:
    for dirpath, dirs, files in os.walk(fasta_folder):
        for file in files:
            file_name = os.path.join(dirpath, file)
            # print(file_name)
            if run in file_name: # Note that there will be ~8 of these
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    for num, line in enumerate(lines):
                        # print(line)
                        if line[0] != ">": # If it's not a header
                            sequence = line # Then it's a sequence
                            # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                            header = metadata_new[metadata_new["Run"] == run].loc[:, "Name"]
                            genotype = metadata_new[metadata_new["Run"] == run].loc[:, "Genotype"]
                            segment = file_name.split("_")[-2]
                            # Find the pair that corresponds to 
                            pair_name = genotype.iloc[0].split("_")[0] + "_" + segment
                            for pair in pairs:
                                # print(pair)
                                # print(pair_name)
                                if pair_name == pair:
                                    fasta_files[pair] = []
                                    fasta_files[pair].append(header)
                                    fasta_files[pair].append(sequence)
                    f.close()

In [ ]:
# Create fasta files 
os.chdir(complete_files)
for pair in fasta_files.keys:
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # First is header, second is sequence
        # print(value)
        output_file.write(item[0] + "\n")
        output_file.write(item[1])
    output_file.close()

In [ ]:

# unique_animals_all = sort_animals_anderson(metadata_new)

# # Flatten unique_animals
# every_unique_animal = []
# for animal in unique_animals_all:
#     every_unique_animal.append(animal)

# print(every_unique_animal)

# unique_animals_set = list(set(every_unique_animal))
# # animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# # animals_df["other"] = unique_animals_set # to sort

# os.chdir(downloads)

# animals_ref = pd.read_csv("animals_ref.csv")


# # If animal not in ref1, put in ref2

# common_animals = []
# # Check if animals in unique_animals_set are in ref1
# for animal in unique_animals_set:
#     for col in animals_ref.columns:
#         if animal in animals_ref[col].values and type(animal) == str:
#             common_animals.append(animal)

# print(common_animals)
# print(len(common_animals))

# different_animals = []
# for animal in unique_animals_set:
#     if animal not in common_animals:
#         different_animals.append(animal)

# print(different_animals)

# # Add to dataframe
# animals_df = animals_ref
# # Make different_animals same length as dataframe, if shorter
# if len(different_animals) < len(animals_df):
#     number_of_times_to_add_nan = len(animals_df) - len(different_animals)
#     for i in range(number_of_times_to_add_nan):
#         different_animals.append(float('nan'))
# # If longer, deal with that later

# animals_df["new"] = (different_animals)

# print(animals_df)

# animals_df.to_csv("animals_ref_to_sort.csv")
